### Game App Deployment

Deploys the Quest Controller Databricks App for Casper's Kitchen Rescue.
This is the mobile-first SPA that serves as the game hub.

In [ ]:
%pip install databricks-sdk --upgrade

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

##### Create SQL warehouse for the quest controller

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

game_warehouse_name = f"{CATALOG}-game-warehouse"

# Reuse any existing warehouse: check for the game warehouse or the default one
warehouse_id = None
for wh in w.warehouses.list():
    if wh.name and wh.name in (game_warehouse_name, f"{CATALOG}-warehouse"):
        warehouse_id = wh.id
        print(f"♻️ Reusing existing warehouse '{wh.name}' (id={wh.id})")
        break

if not warehouse_id:
    warehouse = w.warehouses.create(
        name=game_warehouse_name,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True
    ).result()
    warehouse_id = warehouse.id
    add(CATALOG, "warehouses", warehouse)
    print(f"✅ Created warehouse (id={warehouse_id})")
else:
    print(f"✅ Using warehouse_id={warehouse_id}")

##### Write app.yaml and deploy

In [ ]:
app_yaml_contents = f"""command:
  - uvicorn 
  - app.main:app
env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse_id}'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
"""

import os
yaml_path = "../apps/quest-controller/app.yaml"
if os.path.exists(yaml_path):
    os.remove(yaml_path)

import time
time.sleep(2)

with open(yaml_path, "w") as f:
    f.write(app_yaml_contents)

print(f"\u2705 Wrote app.yaml")

In [ ]:
from databricks.sdk.service.apps import App, AppResource, AppResourceSqlWarehouse, AppResourceSqlWarehouseSqlWarehousePermission
import os

source_code_path = os.path.abspath("../apps/quest-controller")
app_name = f"{CATALOG}-quest-controller"

# Check if app already exists
existing_app = None
try:
    existing_app = w.apps.get(app_name)
    print(f"♻️ App '{app_name}' already exists, will redeploy")
except Exception:
    pass

if not existing_app:
    app = w.apps.create(
        App(
            name=app_name,
            default_source_code_path=source_code_path,
            resources=[
                AppResource(
                    name="sql-warehouse",
                    sql_warehouse=AppResourceSqlWarehouse(
                        id=warehouse_id,
                        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE
                    )
                )
            ]
        )
    )

In [ ]:
if existing_app:
    app_status = existing_app
    print(f"♻️ Using existing app: {app_status.name}")
else:
    app_status = app.result()
    add(CATALOG, "apps", app_status)
    print(f"✅ App created: {app_status.name}")

In [ ]:
# Grant app permissions
from databricks.sdk.service import catalog

# game schema needs MODIFY for INSERT/UPDATE on quest_state, leaderboard, etc.
try:
    w.grants.update(
        full_name=f"{CATALOG}.game",
        securable_type="SCHEMA",
        changes=[
            catalog.PermissionsChange(
                add=[catalog.Privilege.USE_SCHEMA, catalog.Privilege.SELECT, catalog.Privilege.MODIFY],
                principal=app_status.id
            )
        ]
    )
except Exception as e:
    print(f"\u26a0\ufe0f Grant on game: {e}")

# read-only schemas
for schema_name in ["lakeflow", "simulator"]:
    try:
        w.grants.update(
            full_name=f"{CATALOG}.{schema_name}",
            securable_type="SCHEMA",
            changes=[
                catalog.PermissionsChange(
                    add=[catalog.Privilege.USE_SCHEMA, catalog.Privilege.SELECT],
                    principal=app_status.id
                )
            ]
        )
    except Exception as e:
        print(f"\u26a0\ufe0f Grant on {schema_name}: {e}")

w.grants.update(
    full_name=f"{CATALOG}",
    securable_type="CATALOG",
    changes=[
        catalog.PermissionsChange(
            add=[catalog.Privilege.USE_CATALOG],
            principal=app_status.id
        )
    ]
)

print("\u2705 Permissions granted (game: SELECT+MODIFY, lakeflow/simulator: SELECT)")

In [ ]:
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(
        source_code_path=source_code_path
    )
)
deployment_status = deployment.result()

app_url = f"{w.config.host}/apps/{app_status.name}"
print(f"\u2705 Deployed quest controller app: {app_url}")

In [ ]:
# Store app URL in game config for reference
spark.sql(f"""
MERGE INTO {CATALOG}.game.config AS target
USING (SELECT 'quest_app_url' AS config_key, '{app_url}' AS config_value) AS source
ON target.config_key = source.config_key
WHEN MATCHED THEN UPDATE SET config_value = source.config_value
WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
""")

print(f"\n\U0001f3ae Quest Controller App is live!")
print(f"   URL: {app_url}")
print(f"   Open on your phone to start playing.")